# FNI-LLM Training Notebook v2
## Cameroon Language Model — Improved GPU Training

**Improvements over v1:**
- Vocabulary: 500 → 3000 words (fewer UNK tokens)
- Sequence length: 32 → 64 tokens (more context)
- Epochs: 20 → 50 (better convergence)
- Model: larger (256d, 8 heads, 4 layers)
- BPE-style subword tokenization reduces UNK rate

**Languages:** English → French → Bayangi → Douala


In [ ]:
# Cell 1: Mount Drive and sync code
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/FNI_AI_LLM
!git pull origin master

import sys
sys.path.insert(0, '/content/drive/MyDrive/FNI_AI_LLM')
print('Code synced and path set!')

In [ ]:
# Cell 2: Check GPU
!pip install torch --quiet
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
# Cell 3: Improved Tokenizer (larger vocab, fewer UNK)
import re
from collections import Counter

class ImprovedTokenizer:
    PAD, UNK, START, END = '<PAD>', '<UNK>', '<START>', '<END>'

    def __init__(self, max_vocab=3000):
        self.max_vocab = max_vocab
        self.word_to_id = {}
        self.id_to_word = {}
        self.vocab_size = 0

    def tokenize(self, text):
        return re.findall(r"[\w']+|[.,!?;]", text.lower())

    def build_vocab(self, corpus):
        freq = Counter(self.tokenize(corpus))
        words = [w for w, _ in freq.most_common(self.max_vocab - 4)]
        vocab = [self.PAD, self.UNK, self.START, self.END] + words
        self.word_to_id = {w: i for i, w in enumerate(vocab)}
        self.id_to_word = {i: w for w, i in self.word_to_id.items()}
        self.vocab_size = len(vocab)
        unk_rate = sum(1 for w in freq if w not in self.word_to_id) / len(freq)
        print(f'Vocab: {self.vocab_size} | UNK rate: {unk_rate:.1%}')

    def encode(self, text):
        unk = self.word_to_id[self.UNK]
        return [self.word_to_id.get(w, unk) for w in self.tokenize(text)]

    def decode(self, ids):
        skip = {self.PAD, self.UNK, self.START, self.END}
        words = [self.id_to_word.get(i, self.UNK) for i in ids]
        return ' '.join(w for w in words if w not in skip)

print('ImprovedTokenizer defined!')

In [ ]:
# Cell 4: Improved Transformer Model
import torch.nn as nn

class FNITransformerV2(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8,
                 d_ff=1024, num_layers=4, max_seq_len=64, dropout=0.1):
        super().__init__()
        self.embedding    = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout      = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_ff, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm   = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying: share embedding and output weights
        self.output.weight = self.embedding.weight
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        seq_len   = x.shape[1]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.dropout(self.embedding(x) + self.pos_embedding(positions))
        x = self.transformer(x)
        x = self.norm(x)
        return self.output(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print('FNITransformerV2 defined!')

In [ ]:
# Cell 5: Load and prepare data
import numpy as np
import os

LANGUAGE  = 'english'  # Change to: 'french', 'bayangi', 'douala'
SEQ_LEN   = 64
MAX_VOCAB = 3000
DATA_PATH = f'data/cameroon_languages/{LANGUAGE}/processed/{LANGUAGE}_clean.txt'

with open(DATA_PATH, encoding='utf-8') as f:
    corpus = f.read()

tokenizer = ImprovedTokenizer(max_vocab=MAX_VOCAB)
tokenizer.build_vocab(corpus)

# Encode full corpus
all_ids = tokenizer.encode(corpus)
print(f'Total tokens: {len(all_ids):,}')

# Create training samples
samples = []
for i in range(0, len(all_ids) - SEQ_LEN, SEQ_LEN // 2):
    chunk = all_ids[i: i + SEQ_LEN + 1]
    if len(chunk) == SEQ_LEN + 1:
        samples.append(chunk)

np.random.seed(42)
np.random.shuffle(samples)
n = len(samples)
train_samples = samples[:int(n*0.8)]
val_samples   = samples[int(n*0.8):int(n*0.9)]

print(f'Samples - Train: {len(train_samples)} | Val: {len(val_samples)}')

In [ ]:
# Cell 6: Build model and configure training
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 3e-4
WARMUP     = 5

model = FNITransformerV2(
    vocab_size=tokenizer.vocab_size,
    d_model=256, num_heads=8,
    d_ff=1024, num_layers=4,
    max_seq_len=SEQ_LEN
).to(DEVICE)

print(f'Parameters: {model.count_params():,}')
print(f'Device: {DEVICE}')
print(f'Vocab: {tokenizer.vocab_size}')

def make_batches(samples, batch_size, shuffle=True):
    idx = np.arange(len(samples))
    if shuffle: np.random.shuffle(idx)
    for i in range(0, len(idx) - batch_size, batch_size):
        batch = [samples[j] for j in idx[i:i+batch_size]]
        arr   = np.array(batch)
        yield arr[:, :-1], arr[:, 1:]

In [ ]:
# Cell 7: Training loop with warmup scheduler
import time, json

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def get_lr(epoch):
    if epoch < WARMUP:
        return LR * epoch / WARMUP
    progress = (epoch - WARMUP) / (EPOCHS - WARMUP)
    return LR * 0.5 * (1 + np.cos(np.pi * progress))

best_val_loss = float('inf')
log = []
os.makedirs('models/checkpoints', exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    # Update LR
    for g in optimizer.param_groups:
        g['lr'] = get_lr(epoch)

    # Train
    model.train()
    t_losses = []
    t0 = time.time()
    for X_np, y_np in make_batches(train_samples, BATCH_SIZE):
        X = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
        y = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X).reshape(-1, tokenizer.vocab_size), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_losses.append(loss.item())

    # Validate
    model.eval()
    v_losses = []
    with torch.no_grad():
        for X_np, y_np in make_batches(val_samples, BATCH_SIZE, shuffle=False):
            X = torch.tensor(X_np, dtype=torch.long).to(DEVICE)
            y = torch.tensor(y_np, dtype=torch.long).to(DEVICE)
            loss = criterion(model(X).reshape(-1, tokenizer.vocab_size), y.reshape(-1))
            v_losses.append(loss.item())

    tl, vl = np.mean(t_losses), np.mean(v_losses)
    elapsed = time.time() - t0

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save({'model': model.state_dict(),
                    'vocab': tokenizer.word_to_id,
                    'config': {'vocab_size': tokenizer.vocab_size,
                               'd_model': 256, 'num_heads': 8,
                               'd_ff': 1024, 'num_layers': 4,
                               'max_seq_len': SEQ_LEN}},
                   f'models/checkpoints/{LANGUAGE}_v2_best.pt')

    log.append({'epoch': epoch, 'train_loss': round(float(tl), 4),
                'val_loss': round(float(vl), 4), 'time': round(elapsed, 2)})

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | train={tl:.4f} | val={vl:.4f} | '
              f'lr={get_lr(epoch):.2e} | {elapsed:.1f}s')

with open(f'logs/{LANGUAGE}_v2_training.json', 'w') as f:
    json.dump(log, f, indent=2)
print(f'Best val loss: {best_val_loss:.4f}')
print(f'Saved: models/checkpoints/{LANGUAGE}_v2_best.pt')

In [ ]:
# Cell 8: Plot training curves
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

tl = [e['train_loss'] for e in log]
vl = [e['val_loss']   for e in log]

plt.figure(figsize=(10, 4))
plt.plot(tl, label='Train Loss')
plt.plot(vl, label='Val Loss')
plt.title(f'FNI-LLM v2 Training - {LANGUAGE.capitalize()}')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
os.makedirs('docs/visualizations', exist_ok=True)
plt.savefig(f'docs/visualizations/{LANGUAGE}_v2_training_curve.png', dpi=100)
plt.close()
print(f'Saved: docs/visualizations/{LANGUAGE}_v2_training_curve.png')

In [ ]:
# Cell 9: Generate text from trained model
def generate(model, tokenizer, prompt, max_new_tokens=30,
             temperature=0.8, top_k=40, device='cpu'):
    model.eval()
    ids  = tokenizer.encode(prompt)
    seen = {}
    with torch.no_grad():
        for _ in range(max_new_tokens):
            x      = torch.tensor([ids[-SEQ_LEN:]], dtype=torch.long).to(device)
            logits = model(x)[0, -1].cpu().numpy().astype(float)

            # Repetition penalty
            for tid, cnt in seen.items():
                logits[tid] -= 1.5 * cnt

            # Top-k sampling
            logits = logits / temperature
            top_k_idx = np.argsort(logits)[-top_k:]
            top_k_logits = logits[top_k_idx]
            probs = np.exp(top_k_logits - np.max(top_k_logits))
            probs = probs / probs.sum()
            next_id = int(np.random.choice(top_k_idx, p=probs))

            seen[next_id] = seen.get(next_id, 0) + 1
            ids.append(next_id)

    return tokenizer.decode(ids)

# Load best checkpoint
ckpt = torch.load(f'models/checkpoints/{LANGUAGE}_v2_best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])

prompts = [
    'cameroon is a country',
    'the languages of cameroon',
    'douala is the largest',
    'the people of cameroon',
]

print(f'=== GENERATED TEXT ({LANGUAGE.upper()}) ===')
for p in prompts:
    out = generate(model, tokenizer, p, max_new_tokens=20,
                   temperature=0.8, top_k=40, device=DEVICE)
    print(f'Prompt : "{p}"')
    print(f'Output : "{out}"')
    print()

In [ ]:
# Cell 10: Push results to GitHub
!git config --global user.email 'coursdenatationcmr@gmail.com'
!git config --global user.name 'Ronald'
%cd /content/drive/MyDrive/FNI_AI_LLM
!git add logs/ docs/visualizations/
!git commit -m '[Year3] {LANGUAGE} v2 model trained - improved vocab and architecture'
!git push origin master
print('Pushed to GitHub!')